# Execute evalplus samples (HumanEval+ / MBPP+) on Linux

The sandbox that runs model-written code needs Unix signals (`signal.setitimer`, `SIGALRM`), so it cannot run on Windows.
Code is **generated locally** (bf16, pinned model revision) by `run_codegen.py` (in this folder). This notebook only **executes** it.

1. Runtime: CPU is enough (Runtime → Change runtime type → None/CPU).
2. Run the cells. When asked, upload the files `<model>__humaneval.jsonl` and/or `<model>__mbpp.jsonl` from `runs/<model>/evalplus/`.
3. The last cell downloads `evalplus_results.zip`. Put each `<model>__<dataset>_eval_results.json` from it into `runs/<model>/evalplus/`, next to its samples, then run `compare.py`.

In [ ]:
!pip -q install "evalplus==0.3.1"
import evalplus, platform, sys
print("evalplus", evalplus.__version__ if hasattr(evalplus, "__version__") else "0.3.1", "| python", sys.version.split()[0], "|", platform.platform())

In [ ]:
import glob, os
try:
    from google.colab import files
    uploaded = list(files.upload().keys())
except ImportError:  # not on Colab: use files already in the working directory
    uploaded = sorted(glob.glob("*__humaneval.jsonl") + glob.glob("*__mbpp.jsonl"))
samples = [f for f in uploaded if f.endswith(("__humaneval.jsonl", "__mbpp.jsonl"))]
print(samples)
assert samples, "upload <model>__humaneval.jsonl or <model>__mbpp.jsonl"

In [ ]:
import json, subprocess
summary = []
for f in samples:
    dataset = "humaneval" if f.endswith("__humaneval.jsonl") else "mbpp"
    print(f"=== {f} ({dataset})", flush=True)
    out = subprocess.run(["evalplus.evaluate", "--dataset", dataset, "--samples", f, "--parallel", str(os.cpu_count() or 2)],
                         capture_output=True, text=True)
    print(out.stdout[-1500:], out.stderr[-1500:])
    res_path = f.replace(".jsonl", "_eval_results.json")
    res = json.load(open(res_path))
    n = len(res["eval"])
    base = sum(all(r["base_status"] == "pass" for r in v) for v in res["eval"].values()) / n
    plus = sum(all(r["base_status"] == "pass" and r["plus_status"] == "pass" for r in v) for v in res["eval"].values()) / n
    summary.append({"file": f, "dataset": dataset, "n": n, "pass@1_base": round(100 * base, 2), "pass@1_plus": round(100 * plus, 2)})
for s in summary: print(s)
json.dump({"evalplus": "0.3.1", "summary": summary}, open("evalplus_summary.json", "w"), indent=1)

In [ ]:
import zipfile
with zipfile.ZipFile("evalplus_results.zip", "w") as z:
    for f in samples:
        z.write(f.replace(".jsonl", "_eval_results.json"))
    z.write("evalplus_summary.json")
try:
    files.download("evalplus_results.zip")
except NameError:
    print("written evalplus_results.zip")